<a href="https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/03_features_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git
!git -C /content/urban-mobility-forecast pull

Cloning into 'urban-mobility-forecast'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 45 (delta 14), reused 33 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 12.20 KiB | 6.10 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Already up to date.


In [5]:
import sys
import pandas as pd
import importlib.util

In [6]:
sys.path.append('/content/urban-mobility-forecast')

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Load parquet
df = pd.read_parquet("/content/drive/MyDrive/citibike/hourly_demand_final.parquet")

# Load graph module
spec = importlib.util.spec_from_file_location(
    "graph",
    "/content/urban-mobility-forecast/preprocessing/graph.py"
)
graph = importlib.util.module_from_spec(spec)
spec.loader.exec_module(graph)

# Rebuild adjacency matrix to get stations
W, stations = graph.build_adjacency_matrix(df, sigma2=0.001, theta=0.5)
print(f"Stations: {len(stations)}")

Mounted at /content/drive
Stations     : 438
Non-zero edges: 55988
Sparsity     : 70.82%
Stations: 438


In [7]:
# Load modules directly
spec = importlib.util.spec_from_file_location(
    "features",
    "/content/urban-mobility-forecast/preprocessing/features.py"
)
features = importlib.util.module_from_spec(spec)
spec.loader.exec_module(features)

# Build demand matrix
station_ids = stations['start_station_id'].tolist()
demand_matrix, hours = features.build_demand_matrix(df, station_ids)
print(f"demand_matrix: {demand_matrix.shape}")

# Normalize
normalized, scaler = features.normalize_demand(demand_matrix)
print(f"normalized: {normalized.shape}, range: [{normalized.min():.2f}, {normalized.max():.2f}]")

# Time features
time_feats = features.build_time_features(pd.Series(hours))
print(f"time_features: {time_feats.shape}")

# Sliding windows
x_demand, x_time, y = features.build_sliding_windows(normalized, time_feats)

demand_matrix: (16813, 438)
normalized: (16813, 438), range: [-1.00, 1.00]
time_features: (16813, 6)
Samples  : 16670
x_demand : (16670, 438, 72, 1)
x_time   : (16670, 72, 6)
y        : (16670, 438, 72, 1)
